# A sparse 1D field with three long-range connections

Build a **128-site ring**, add three directed long-range connections, perturb a
small patch, and watch the response spread locally and jump to distant sites.
Then explore **excitability, local coupling, and non-local coupling** in a
128-combination parameter sweep. No external data are needed.

We use TVB's hybrid API with one `Subnetwork` and sparse `IntraProjection`s.
This is a finite network on a periodic 1D geometry, not a continuum PDE.

Here *subcritical* means **below the network's oscillation threshold**.
`SupHopf` has a supercritical Hopf bifurcation; this example does not model a
subcritical Hopf bifurcation, bistability, or hysteresis.

## 1. Setup

Use **Python 3.12 or newer**. This installation matches the site's native runtime
pin in `requirements/base.txt` and includes the C++ backend used for the sweep:

```bash
python3.12 -m venv .venv
source .venv/bin/activate  # Windows: .venv\Scripts\activate
python -m pip install --extra-index-url https://ins-amu.github.io/tvb-wheels/simple/ \
    "tvb-library==3.0.0a1.post3" numpy scipy matplotlib ipykernel
```

Select that environment as the notebook kernel and **Run All**. An older TVB
release may not contain these experimental APIs. If installation fails, see
[installation](../../getting-started/installation.md). Native Python is the
reference environment here; browser execution needs a matching backend build.

Time is in **ms**, position in **mm**, and activity in arbitrary model units.
These are illustrative coefficients, not a fit to cortical data. Runs are
deterministic, with explicit initial states and no noise.

In [ ]:
%matplotlib inline
from importlib.metadata import version
from time import perf_counter
import numpy as np
import scipy.sparse as sp
from scipy.linalg import eigvals
import matplotlib.pyplot as plt

from tvb.simulator.models.oscillator import SupHopf
from tvb.simulator.integrators import HeunDeterministic
from tvb.simulator.monitors import Raw
from tvb.simulator.hybrid import Subnetwork, IntraProjection, NetworkSet, Simulator
from tvb.simulator.hybrid.coupling import Linear as HybridLinear

try:
    from tvb.simulator.backend.cpp_hybrid import notebook as cpp_nb
except ImportError as exc:
    raise ImportError(
        "Install the TVB version above in Python >=3.12, then restart the kernel. "
        "This notebook requires the C++ sweep backend."
    ) from exc

print(f"tvb-library {version('tvb-library')}")

## 2. Local neighbours and exactly three non-local connections

Place $N=128$ sites 1 mm apart on a periodic line of circumference 128 mm.
Each target receives from the three nearest neighbours on each side. Weights
fall off as a Gaussian with width 1.5 mm and are normalized so each **local**
row sums to 1. There are no self-edges.

Keep the three shortcuts in a separate matrix. Written as **source → target**,
they form a directed long-range loop:

**64 → 16, 16 → 104, 104 → 64.**

Each shortcut has weight 1 before applying the non-local gain. They are **three
directed edges**, not three bidirectional pairs. Do not renormalize the combined
matrix: the local and non-local gains should remain independent controls.
Rows are targets and columns are sources in both sparse matrices.

In [ ]:
N = 128
DX = 1.0
CIRCUMFERENCE = N * DX
KERNEL_WIDTH = 1.5
NEIGHBOURS = 3
position = np.arange(N) * DX
SHORTCUTS = [(64, 16), (16, 104), (104, 64)]  # (source, target)

rows, cols, values = [], [], []
for target in range(N):
    for offset in range(-NEIGHBOURS, NEIGHBOURS + 1):
        if offset != 0:
            rows.append(target)
            cols.append((target + offset) % N)
            values.append(np.exp(-0.5 * (offset * DX / KERNEL_WIDTH)**2))

W_local = sp.csr_matrix((values, (rows, cols)), shape=(N, N), dtype=np.float64)
W_local = (sp.diags(1 / np.asarray(W_local.sum(axis=1)).ravel()) @ W_local).tocsr()
W_local.sort_indices()
W_nonlocal = sp.csr_matrix(
    (np.ones(len(SHORTCUTS)), ([target for source, target in SHORTCUTS],
                              [source for source, target in SHORTCUTS])),
    shape=(N, N), dtype=np.float64,
)

assert W_local.nnz == N * 2 * NEIGHBOURS
assert W_nonlocal.nnz == 3
assert W_local.multiply(W_nonlocal).nnz == 0
assert np.all(W_local.diagonal() == 0) and np.all(W_nonlocal.diagonal() == 0)
np.testing.assert_allclose(np.asarray(W_local.sum(axis=1)).ravel(), 1.0)
np.testing.assert_allclose(W_local.toarray(), W_local.T.toarray(), atol=1e-14)
for source, target in SHORTCUTS:
    assert W_nonlocal[target, source] == 1 and W_nonlocal[source, target] == 0
    assert min(abs(target-source), N-abs(target-source)) > NEIGHBOURS
print(f"{N} sites; {W_local.nnz} local + {W_nonlocal.nnz} non-local directed edges")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), layout="constrained")
theta = 2 * np.pi * position / CIRCUMFERENCE
circle_x, circle_y = np.cos(theta), np.sin(theta)
for i, j in zip(*W_local.nonzero()):
    if i < j:
        axes[0].plot(circle_x[[i, j]], circle_y[[i, j]], color="0.85", lw=0.5, zorder=0)
axes[0].scatter(circle_x, circle_y, s=9, color="#275b97")
for source, target in SHORTCUTS:
    axes[0].annotate("", xy=(circle_x[target], circle_y[target]),
                     xytext=(circle_x[source], circle_y[source]),
                     arrowprops=dict(arrowstyle="->", color="#c34b20", lw=1.6))
for i in sorted({0, *[node for edge in SHORTCUTS for node in edge]}):
    axes[0].text(1.15*circle_x[i], 1.15*circle_y[i], str(i), ha="center", va="center")
axes[0].set(aspect="equal", title="Local neighbours + three directed shortcuts",
            xlim=(-1.3, 1.3), ylim=(-1.3, 1.3))
axes[0].axis("off")
axes[1].spy(W_local, markersize=1, color="#275b97", label="Local", origin="lower")
axes[1].scatter([s for s, t in SHORTCUTS], [t for s, t in SHORTCUTS],
                color="#c34b20", s=35, marker="x", label="Non-local")
axes[1].set(xlabel="Source site", ylabel="Target site", title="Sparse connection pattern")
axes[1].legend(frameon=False)
plt.show()

## 3. Excitability and the two coupling strengths

Each site's state is $z_i=x_i+\mathrm{i}y_i$. The model is

$$
\dot z_i=(a+\mathrm{i}\omega-|z_i|^2)z_i
  +K_{\mathrm L}\sum_j W^{\mathrm L}_{ij}z_j
  +K_{\mathrm N}\sum_j W^{\mathrm N}_{ij}z_j.
$$

- **Excitability $a$:** a negative value damps an isolated site; increasing it
  makes sustained activity easier.
- **Local gain $K_{\mathrm L}$:** controls spread through nearby sites.
- **Non-local gain $K_{\mathrm N}$:** controls transmission around the three
  long-range connections.

Coupling is additive, not a difference between source and target activity.
Both $x$ and $y$ receive the same two pathways. We use four projections: local
and non-local inputs to each state component. Projection names `local` and
`long_range` group the two components so each pathway has a single sweep control.
Each projection's `scale` stays at 1; its `Linear.a` holds that pathway's gain.

### Check the network threshold, not just the isolated model

Let $C=K_{\mathrm L}W^{\mathrm L}+K_{\mathrm N}W^{\mathrm N}$ and let $\rho(C)$ be
its largest eigenvalue. For these nonnegative matrices this is also the largest
real part of any eigenvalue. The zero state is linearly stable when

$$\mu=a+\rho(C)<0.$$

Without shortcuts, $\rho(C)=K_{\mathrm L}$ and the familiar boundary is
$a+K_{\mathrm L}=0$. **With shortcuts, that line no longer applies.** The leading
spatial pattern need not be uniform, so neither does the simple uniform-amplitude
formula. We calculate the stability boundary for each pair of gains instead.

| Setting | Baseline | Meaning |
|---|---:|---|
| Excitability $a$ | $-0.012$ ms$^{-1}$ | Isolated sites are damped |
| Local gain $K_{\mathrm L}$ | $0.008$ ms$^{-1}$ | Nearby recruitment |
| Non-local gain $K_{\mathrm N}$ | $0.006$ ms$^{-1}$ | Weak distant recruitment |
| Frequency | 10 Hz | $\omega=2\pi\,10/1000$ rad/ms |
| Step | 0.2 ms | 500 steps per cycle |
| Tract lengths | 0 mm | No physical conduction delay |

Below we verify that the **whole baseline network** is below onset. The cubic
term limits growth above onset; the activity units are not calibrated voltages.

In [ ]:
DT = 0.2
OMEGA = 2 * np.pi * 10 / 1000
A_BASE, LOCAL_BASE, NONLOCAL_BASE = -0.012, 0.008, 0.006
BASE_DURATION = 1200.0
BASE_STEPS = int(round(BASE_DURATION / DT))

# Small dense matrices are used only for this stability diagnostic.
local_dense, nonlocal_dense = W_local.toarray(), W_nonlocal.toarray()
def coupling_growth(local_gain, nonlocal_gain):
    return eigvals(local_gain*local_dense + nonlocal_gain*nonlocal_dense).real.max()

np.testing.assert_allclose(coupling_growth(LOCAL_BASE, 0), LOCAL_BASE, atol=1e-12)
np.testing.assert_allclose(coupling_growth(0, NONLOCAL_BASE), NONLOCAL_BASE, atol=1e-12)
mu_base = A_BASE + coupling_growth(LOCAL_BASE, NONLOCAL_BASE)
assert mu_base < 0
print(f"Baseline growth rate: {mu_base:.5f} /ms; slowest linear decay time: {-1/mu_base:.0f} ms")


def make_field(a=A_BASE, local_gain=LOCAL_BASE, nonlocal_gain=NONLOCAL_BASE, dt=DT):
    """Fresh model, integrator, projections and history for every independent run."""
    field = Subnetwork(
        name="field", nnodes=N, node_indices=np.arange(N),
        model=SupHopf(a=np.array([a]), omega=np.array([OMEGA]), variables_of_interest=("x", "y")),
        scheme=HeunDeterministic(dt=dt),
    )
    field.projections = []
    for name, weights, gain in (("local", W_local, local_gain),
                                ("long_range", W_nonlocal, nonlocal_gain)):
        for slot in (0, 1):
            projection = IntraProjection(
                source_cvar=np.array([slot], dtype=np.int_),
                target_cvar=np.array([slot], dtype=np.int_),
                weights=weights.copy(), lengths=sp.csr_matrix((N, N)),
                cv=1.0, dt=dt, scale=1.0,
                cfun=HybridLinear(a=np.array([gain]), b=np.array([0.0])),
            )
            projection.name = name  # Same pathway name deliberately covers both x and y.
            field.projections.append(projection)
    return NetworkSet(subnets=[field], projections=[]).configure()


centre = N // 2
ring_distance = np.minimum(abs(position-position[centre]),
                           CIRCUMFERENCE-abs(position-position[centre]))
initial_state = np.zeros((2, N, 1))  # (state component, site, mode)
initial_state[0, :, 0] = 0.08 * np.exp(-0.5 * (ring_distance / 1.5)**2)
# A displacement at t=0, not ongoing stimulation. Exact zero would stay zero,
# even above onset. Every sweep member receives this same nonzero seed.

## 4. A localized perturbation with distant echoes

Run the below-onset baseline first using Python as a reference. The monitor
returns `(time, state component, site, mode)`. Watch the seed at site 64 and the
distant sites 16 and 104: the shortcuts recruit those sites without waiting for
activity to spread along all the intermediate neighbours.

In [ ]:
reference = Simulator(nets=make_field(), backend="python",
                      simulation_length=BASE_DURATION, monitors=[Raw()])
reference.configure()
((t_ref, y_ref),) = reference.run(initial_conditions=[initial_state.copy()])
assert y_ref.shape == (BASE_STEPS, 2, N, 1) and np.isfinite(y_ref).all()
x_ref = y_ref[:, 0, :, 0]
r_ref = np.hypot(y_ref[:, 0, :, 0], y_ref[:, 1, :, 0])
assert r_ref[-1].max() < 0.02 * initial_state[0, :, 0].max()

sites = [64, 16, 104]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3), layout="constrained")
image = axes[0].pcolormesh(t_ref, position, x_ref.T, shading="auto", cmap="RdBu_r", vmin=-0.08, vmax=0.08)
axes[0].set(xlabel="Time (ms)", ylabel="Position (mm)", title="Local response and distant echoes")
fig.colorbar(image, ax=axes[0], label="x (a.u.)")
for site in sites:
    axes[1].plot(t_ref, x_ref[:, site], label=f"Site {site}", lw=1)
axes[1].set(xlabel="Time (ms)", ylabel="x (a.u.)", title="Seed and shortcut endpoints")
axes[1].legend(frameon=False)
for time_ms in (0, 100, 400, 1000):
    profile = initial_state[0, :, 0] if time_ms == 0 else r_ref[np.argmin(abs(t_ref-time_ms))]
    axes[2].plot(position, profile, label=f"{time_ms} ms")
axes[2].set(xlabel="Position (mm)", ylabel="Radius r (a.u.)", title="Spatial envelope")
axes[2].legend(frameon=False)
plt.show()

These are **transient**, not stationary, oscillations: the unforced baseline
relaxes to zero. The distant response is not a measurement of axonal speed;
tract lengths are zero in this example.

## 5. A short numerical check

Compare the reference with the C++ sweep at the same parameters, then halve the
step size. Small differences are expected from numerical precision. For temporal
averaging, `chunk_size` is the window in **integration steps**: a value of 1
retains every step. `monitor_period` is not the averaging-window control here.
Use pathway-specific names, not the global `coupling_scale` alias, to keep local
and non-local coupling independent.

In [ ]:
base_params = {"field.a": np.array([A_BASE]), "local.a": np.array([LOCAL_BASE]),
               "long_range.a": np.array([NONLOCAL_BASE])}
cpp_check = cpp_nb.sweep(make_field(), BASE_STEPS, initial_states=[initial_state.copy()],
                         monitor="tavg", chunk_size=1, workers=1, progress=False, **base_params)
# Remove the mode axis, retaining (run, time, state component, site).
y_cpp = cpp_check.squeezed("field")[0]
assert cpp_check.backend.startswith("cpp-simd")
np.testing.assert_allclose(cpp_check.times, t_ref, rtol=1e-6, atol=1e-4)
np.testing.assert_allclose(y_cpp, y_ref[..., 0], rtol=2e-3, atol=2e-5)

fine_check = cpp_nb.sweep(make_field(dt=DT/2), 2*BASE_STEPS,
                          initial_states=[initial_state.copy()], monitor="tavg", chunk_size=1,
                          workers=1, progress=False, **base_params)
y_fine = fine_check.squeezed("field")[0, 1::2]  # Match the original end-of-step times.
np.testing.assert_allclose(y_cpp, y_fine, rtol=2e-2, atol=2e-4)
print(f"Max difference: Python vs C++ {np.max(abs(y_cpp-y_ref[..., 0])):.2e}; "
      f"step vs half-step {np.max(abs(y_cpp-y_fine)):.2e}")

## 6. Sweep excitability, local coupling, and non-local coupling

Use **8 excitabilities × 4 local gains × 4 non-local gains = 128 runs**, each on
128 spatial sites. `meshgrid` constructs the full product: the sweep itself pairs
parameter arrays elementwise. Every run restarts from the same seed, rather than
continuing from the preceding run's final state.

Integrate for 4000 ms, save 5 ms averages, and use the final 2000 ms for analysis.
Two workers limit CPU use. The state and coupling outputs together occupy about
200 MiB, plus analysis arrays and overhead.

**Timing:** time the entire sweep call, including its setup and output collection,
but not network construction, plotting, or stability calculations. Report wall
time and wall time divided by **runs × steps × nonzero projection weights**.
There are 771 directed graph edges and **1542 projection weights** because each
edge acts on both $x$ and $y$. The denominator includes zero-gain pathways at
parameter points where a gain is zero, but excludes zero-valued sparse padding.
This is an end-to-end throughput summary, not an isolated coupling-operation
benchmark or a count of integrator derivative evaluations.

In [ ]:
a_values = np.linspace(-0.024, 0.004, 8, dtype=np.float32)
local_values = np.array([0.0, 0.004, 0.008, 0.012], dtype=np.float32)
nonlocal_values = np.array([0.0, 0.008, 0.016, 0.024], dtype=np.float32)
a_grid, local_grid, nonlocal_grid = np.meshgrid(a_values, local_values, nonlocal_values, indexing="ij")
params = {"field.a": a_grid.ravel(), "local.a": local_grid.ravel(),
          "long_range.a": nonlocal_grid.ravel()}
SWEEP_DURATION, BURN_IN = 4000.0, 2000.0
MONITOR_STEPS = 25               # 5 ms at dt=0.2 ms
n_steps = int(round(SWEEP_DURATION / DT))
n_runs = a_grid.size
sweep_network = make_field()
projection_nnz = sum(p.weights.count_nonzero() for sn in sweep_network.subnets for p in sn.projections)
assert n_runs == 128 and projection_nnz == 2*(W_local.nnz + W_nonlocal.nnz)

start = perf_counter()
result = cpp_nb.sweep(sweep_network, n_steps, initial_states=[initial_state.copy()],
                      workers=2, progress=False, monitor="tavg", chunk_size=MONITOR_STEPS, **params)
sweep_wall = perf_counter() - start
ns_per_nnz_step = sweep_wall * 1e9 / (n_runs * n_steps * projection_nnz)
print(f"Sweep: {n_runs} runs × {n_steps:,} steps; {projection_nnz:,} nonzero x/y weights")
print(f"Wall time: {sweep_wall:.3f} s; {ns_per_nnz_step:.3f} ns / (nnz × step × run)")

trajectory = result.squeezed("field")
assert trajectory.shape == (n_runs, n_steps//MONITOR_STEPS, 2, N)
assert np.isfinite(trajectory).all() and result.backend.startswith("cpp-simd")
np.testing.assert_allclose(result.sweep_values, np.column_stack(list(params.values())))
np.testing.assert_allclose(np.diff(result.times), MONITOR_STEPS*DT, rtol=1e-6)
np.testing.assert_allclose(result.times[0], (1+MONITOR_STEPS)*DT/2, rtol=1e-6)

### Activity maps, one panel per non-local gain

A signed time average of $x$ would cancel positive and negative oscillations.
Instead calculate the radius $r=\sqrt{x^2+y^2}$ and its **root-mean-square over
sites and the late time window**. This includes localized activity at shortcut
endpoints; it does not assume the whole ring oscillates uniformly. The 5 ms
averaging window slightly attenuates a 100 ms cycle.

The dashed curves mark the **linear** threshold $a+\rho(C)=0$. We calculate one
spectrum for each of the 16 gain pairs and reuse it across excitabilities.
Near a threshold, growth or decay can be too slow for four seconds to represent
the long-time behaviour. Sparse shortcuts can also concentrate activity in a
small part of the ring: colour intensity is not a binary stability classifier.

In [ ]:
radius = np.hypot(trajectory[:, :, 0, :], trajectory[:, :, 1, :])
late = result.times >= BURN_IN
assert late.any() and (~late).any()
late_rms = np.sqrt(np.mean(radius[:, late, :]**2, axis=(1, 2))).reshape(a_grid.shape)
gain_growth = np.array([[coupling_growth(float(local), float(nonlocal_gain))
                         for nonlocal_gain in nonlocal_values] for local in local_values])
mu_grid = a_grid + gain_growth[None, :, :]
# Check away from the threshold; finite-time activity near it is not a failure.
assert late_rms[mu_grid < -0.004].max() < 1e-4
assert late_rms[mu_grid > 0.008].min() > 1e-3

fig, axes = plt.subplots(1, 4, figsize=(12, 3.4), sharey=True, layout="constrained")
for j, ax in enumerate(axes):
    image = ax.pcolormesh(local_values, a_values, late_rms[:, :, j], shading="nearest",
                          cmap="magma", vmin=0, vmax=late_rms.max())
    ax.plot(local_values, -gain_growth[:, j], "--", color="cyan", label="Linear onset")
    ax.set(xlabel="Local gain (1/ms)", title=f"Non-local gain = {nonlocal_values[j]:.3f}",
            ylim=(a_values[0]-0.002, a_values[-1]+0.002))
axes[0].set_ylabel("Excitability a (1/ms)")
axes[0].legend(frameon=False, loc="upper right")
fig.colorbar(image, ax=axes, label="Late RMS radius (a.u.)", shrink=0.85)
plt.show()

## 7. What do the shortcuts change?

Hold excitability at $a=-0.012$ and local gain at 0.008, and compare no, weak,
and strong non-local coupling. The first two networks are below onset; strong
shortcuts make the recurrent long-range loop self-sustaining even though each
isolated site is damped.

The space–time panels and node traces show the first 1000 ms so cycles remain
visible. The final envelope panel uses all 4000 ms. Watch how the endpoints
16 and 104 become active before the intervening sites do.

In [ ]:
selected_ijk = [(3, 2, 0), (3, 2, 1), (3, 2, 3)]
selected = [np.ravel_multi_index(ijk, a_grid.shape) for ijk in selected_ijk]
labels = ["No shortcuts", "Weak shortcuts", "Strong shortcuts"]
colours = ["#275b97", "#777777", "#9e4830"]
assert all(mu_grid[ijk] < 0 for ijk in selected_ijk[:2]) and mu_grid[selected_ijk[2]] > 0

fig, axes = plt.subplots(2, 3, figsize=(12, 6), layout="constrained")
early = result.times <= 1000
for col, (member, label) in enumerate(zip(selected, labels)):
    a, local_gain, nonlocal_gain = result.sweep_values[member]
    image = axes[0, col].pcolormesh(result.times[early], position,
                                   trajectory[member, early, 0, :].T,
                                   shading="auto", cmap="RdBu_r", vmin=-0.12, vmax=0.12)
    axes[0, col].set(title=f"{label}: non-local gain={nonlocal_gain:.3f}",
                     xlabel="Time (ms)", ylabel="Position (mm)")
    for site in sites:
        axes[1, col].plot(result.times[early], trajectory[member, early, 0, site],
                          label=f"Site {site}", lw=0.9)
    axes[1, col].set(xlabel="Time (ms)", ylabel="x (a.u.)")
axes[1, 0].legend(frameon=False)
fig.colorbar(image, ax=axes[0, :], label="x (a.u.; shared scale)", shrink=0.85)
plt.show()

fig, ax = plt.subplots(figsize=(8, 3), layout="constrained")
for member, label, colour in zip(selected, labels, colours):
    ax.semilogy(result.times, np.sqrt(np.mean(radius[member]**2, axis=1)), label=label, color=colour)
ax.axvline(BURN_IN, color="0.5", ls=":", label="Start of analysis window")
ax.set(xlabel="Time (ms)", ylabel="Spatial RMS radius (a.u.)", title="Damped echoes or sustained recruitment?")
ax.legend(frameon=False, ncol=2)
plt.show()

## 8. Checks and next steps

The notebook checks the 128-site graph, all three shortcut directions, independent
pathway names, baseline stability, finite output, parameter ordering, Python/C++
agreement, and a baseline step-size refinement. Away from onset it also checks
that stable runs decay and sufficiently unstable runs develop activity.

- **Resolve onset more closely:** refine the excitability and gain values, extend
  simulation and burn-in together, and halve the integration step while keeping
  the saved interval in ms unchanged. The baseline step check does not establish
  convergence everywhere in the sweep.
- **Compare performance:** keep the same timing denominator, step size, saved
  interval, worker count, and machine. Increasing the site count with fixed
  spacing increases circumference, not spatial resolution at fixed ring size.
- **Move the shortcuts:** edit `SHORTCUTS`, then recalculate the spectra. The
  current three-edge loop is intentional; other directions can produce mostly
  transient, rather than recurrent, distant recruitment.
- **Add conduction delays:** give the connected edges tract lengths and choose
  a velocity in mm/ms. The delay-free threshold used here then needs replacing.
- **Study ongoing below-onset fluctuations:** add an explicit drive or noise
  model; an initial displacement alone always decays in the baseline regime.

Continue with [API contracts](hybrid-contracts.md),
[backends and parameter sweeps](hybrid-backends-and-sweeps.md), and
[heterogeneous hybrid tutorials](hybrid-tutorials.md).